# QAOA Ising Depth Sweep -- Colab Driver

Driver notebook: installs pinned dependencies, clones this repo (on
Colab), and calls straight into `src/experiment.py`'s `run_depth_sweep`.

This notebook also runs when opened locally (outside Colab) against an
setup repo with `requirements.txt` installed -- the setup
cell below detects the environment and skips the pip-install/clone step
in that case.

## Setup

**On Colab**: for the GPU device path, select **Runtime -> Change runtime
type -> GPU** first, then set `COLAB_DEVICE = "GPU"` in the cell below
before running it. This cell always uninstalls both `qiskit-aer` variants
before installing the one you asked for, so it's safe to re-run any time.

In [1]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/raumsie/qaoa-ising.git"
COLAB_DEVICE = "GPU"  # "GPU" for the CUDA-enabled qiskit-aer-gpu build

# qiskit-aer-gpu has no release compatible with qiskit 2.x
PINS = {
    "GPU": {"qiskit": "1.1.0", "aer_pkg": "qiskit-aer-gpu", "aer": "0.15.1"},
    "CPU": {"qiskit": "2.5.0", "aer_pkg": "qiskit-aer", "aer": "0.17.2"},
}[COLAB_DEVICE]

if IN_COLAB:
    _already_imported = [m for m in ("qiskit", "qiskit_aer") if m in sys.modules]

    !pip uninstall -y -q qiskit-aer qiskit-aer-gpu
    !pip install -q qiskit=={PINS["qiskit"]} {PINS["aer_pkg"]}=={PINS["aer"]} qiskit-algorithms==0.4.0 scipy==1.18.0 matplotlib==3.11.1

    if _already_imported:
        print(
            f"{', '.join(_already_imported)} was already imported before this "
            f"install, so the kernel still holds the previous qiskit version.\n"
            "Restarting the runtime -- RE-RUN THIS CELL once it comes back."
        )
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)
        raise SystemExit("Restarting runtime; re-run this cell.")

    import qiskit
    import qiskit_aer

    # Verify on-disk install actually matches the pins
    problems = []
    if qiskit.__version__ != PINS["qiskit"]:
        problems.append(f"qiskit {qiskit.__version__} != pinned {PINS['qiskit']}")
    if qiskit_aer.__version__ != PINS["aer"]:
        problems.append(f"qiskit-aer {qiskit_aer.__version__} != pinned {PINS['aer']}")
    if problems:
        raise RuntimeError(
            "Dependency mismatch for COLAB_DEVICE=" + COLAB_DEVICE + ": "
            + "; ".join(problems)
            + ". Restart the runtime (Runtime > Restart session) and re-run this cell."
        )

    print(f"qiskit {qiskit.__version__} / qiskit_aer {qiskit_aer.__version__} ready (COLAB_DEVICE={COLAB_DEVICE})")

    # Clone/cd via an absolute path rather than %cd's relative check
    CLONE_PARENT = "/content"
    repo_root = os.path.join(CLONE_PARENT, "qaoa-ising")
    if not os.path.isdir(repo_root):
        os.chdir(CLONE_PARENT)
        !git clone {REPO_URL}
    os.chdir(repo_root)
else:
    # Local run: repo is already checked out and requirements.txt already
    # installed in the current environment --
    # just make sure `src` can be found whether running
    # from the repo root or from notebooks/.
    def _find_repo_root(start):
        d = os.path.abspath(start)
        for _ in range(5):
            if os.path.isdir(os.path.join(d, "src")) and os.path.isfile(os.path.join(d, "requirements.txt")):
                return d
            d = os.path.dirname(d)
        raise RuntimeError(f"Could not locate qaoa-ising repo root from {start}")

    repo_root = _find_repo_root(os.getcwd())
    print(
        "Not running in Colab -- assuming requirements.txt is already "
        "installed in the current environment; skipping pip install/git clone."
    )

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"IN_COLAB={IN_COLAB}, repo_root=./{os.path.basename(repo_root)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 83.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 20.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 87.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 MB 16.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 10.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 117.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 11.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB

In [2]:
from src.experiment import run_depth_sweep, records_to_dicts
from src.ising_model import generate_test_instances

## Run the depth sweep

Sweeps every (instance) x (QAOA depth `p`) x (optimizer) combination
via `run_depth_sweep` and records `epsilon(p) = (E_qaoa - E_0)/|E_0|`
against the exact-diagonalization baseline for each point.

The constants below are deliberately small for testing. Increase them at your leisure.

In [3]:
import json
import time

# MINIMIZE_OPTIONS caps optimizer cost for a fast first run (maxiter for
# COBYLA, maxfun for L-BFGS-B). Set to None for an uncapped run.
P_VALUES = range(1, 3)
N_RESTARTS = 3                                # full sweep: 5
MINIMIZE_OPTIONS = {"maxiter": 30, "maxfun": 30}  # full sweep: None
OPTIMIZER_METHODS = ("COBYLA", "L-BFGS-B")
DEVICE = COLAB_DEVICE                          # setup cell's install choice
SEED = 104
N_SPINS = 6                                   # (qubits) generate_test_instances default

instances = generate_test_instances(n_spins=N_SPINS)
print("Instances:", list(instances.keys()))

t0 = time.time()
records = run_depth_sweep(
    instances=instances,
    p_values=P_VALUES,
    optimizer_methods=OPTIMIZER_METHODS,
    device=DEVICE,
    n_restarts=N_RESTARTS,
    minimize_options=MINIMIZE_OPTIONS,
    seed=SEED,
    verbose=True,
)
elapsed = time.time() - t0
print(f"\nSweep complete: {len(records)} records in {elapsed:.1f}s (device={DEVICE})")

Instances: ['uniform_FM', 'frustrated', 'with_field', 'frustrated_pbc']
[uniform_FM] p=1 COBYLA: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (1.05s, 90 fevals)
[uniform_FM] p=1 L-BFGS-B: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (0.63s, 96 fevals)
[uniform_FM] p=2 COBYLA: best_energy=-3.387688 E_0=-5.000000 epsilon=0.322462 (0.79s, 90 fevals)
[uniform_FM] p=2 L-BFGS-B: best_energy=-3.425849 E_0=-5.000000 epsilon=0.314830 (0.84s, 110 fevals)
[frustrated] p=1 COBYLA: best_energy=-0.441817 E_0=-2.593732 epsilon=0.829660 (0.63s, 90 fevals)
[frustrated] p=1 L-BFGS-B: best_energy=-0.936141 E_0=-2.593732 epsilon=0.639076 (0.77s, 123 fevals)
[frustrated] p=2 COBYLA: best_energy=-1.853675 E_0=-2.593732 epsilon=0.285325 (0.86s, 90 fevals)
[frustrated] p=2 L-BFGS-B: best_energy=-1.109949 E_0=-2.593732 epsilon=0.572065 (0.93s, 105 fevals)
[with_field] p=1 COBYLA: best_energy=-2.432381 E_0=-5.632835 epsilon=0.568178 (1.16s, 90 fevals)
[with_field] p=1 L-BFGS-B: best_energy=-2.00

## Save results

Saved as JSON (via `records_to_dicts`) under `results/` so
`notebooks/analysis.ipynb` can load and plot them without recomputing
anything (and without any Colab dependency).

In [4]:
import qiskit

RESULTS_DIR = os.path.join(repo_root, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

output = {
    "config": {
        "p_values": list(P_VALUES),
        "n_restarts": N_RESTARTS,
        "optimizer_methods": list(OPTIMIZER_METHODS),
        "minimize_options": MINIMIZE_OPTIONS,
        "device": DEVICE,
        "seed": SEED,
        "n_spins": N_SPINS,
        "instance_names": list(instances.keys()),
        "wall_time_s_total": elapsed,
        "qiskit_version": qiskit.__version__,  # differs by device: 1.1.0 for GPU, 2.5.0 for CPU
    },
    "records": records_to_dicts(records),
}

results_path = os.path.join(RESULTS_DIR, "depth_sweep_results.json")
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(records)} records to {os.path.relpath(results_path, repo_root)}")

Saved 32 records to results/depth_sweep_results.json


## Optional: CPU-vs-GPU wall-clock comparison

Runs a small matched subset of the sweep on `device="GPU"` and
`device="CPU_AER"` and saves both timings so `analysis.ipynb` can show a
CPU-vs-GPU bar chart. `CPU_AER` (not the sweep's default `"CPU"`) is used
deliberately here: `"CPU"` is qiskit's own plain reference simulator
(`StatevectorEstimator`), a completely different codebase from Aer's
GPU backend, so comparing it against `"GPU"` would measure two different
simulator engines, not hardware. `CPU_AER` runs the identical Aer
`EstimatorV2` code path as `"GPU"`, just with `AerSimulator(device="CPU")`.

This only works on a Colab GPU runtime with `qiskit-aer-gpu` installed.

**Must run the main sweep before this cell**

In [5]:
GPU_TIMING_P_VALUES = range(1, 4)
GPU_TIMING_N_RESTARTS = 2
GPU_TIMING_MINIMIZE_OPTIONS = {"maxiter": 50, "maxfun": 60}
GPU_TIMING_N_SPINS = N_SPINS                   # edit to size this test independently of the main sweep
GPU_TIMING_INSTANCES = {"uniform_FM": generate_test_instances(n_spins=GPU_TIMING_N_SPINS)["uniform_FM"]}

gpu_results_path = os.path.join(RESULTS_DIR, "gpu_timing_results.json")

try:
    t0 = time.time()
    gpu_records = run_depth_sweep(
        instances=GPU_TIMING_INSTANCES,
        p_values=GPU_TIMING_P_VALUES,
        optimizer_methods=OPTIMIZER_METHODS,
        device="GPU",
        n_restarts=GPU_TIMING_N_RESTARTS,
        minimize_options=GPU_TIMING_MINIMIZE_OPTIONS,
        seed=SEED,
        verbose=True,
    )
    gpu_elapsed = time.time() - t0

    t0 = time.time()
    cpu_records = run_depth_sweep(
        instances=GPU_TIMING_INSTANCES,
        p_values=GPU_TIMING_P_VALUES,
        optimizer_methods=OPTIMIZER_METHODS,
        device="CPU_AER",
        n_restarts=GPU_TIMING_N_RESTARTS,
        minimize_options=GPU_TIMING_MINIMIZE_OPTIONS,
        seed=SEED,
        verbose=True,
    )
    cpu_elapsed = time.time() - t0

    gpu_output = {
        "config": {
            "p_values": list(GPU_TIMING_P_VALUES),
            "n_restarts": GPU_TIMING_N_RESTARTS,
            "minimize_options": GPU_TIMING_MINIMIZE_OPTIONS,
            "optimizer_methods": list(OPTIMIZER_METHODS),
            "instance_names": list(GPU_TIMING_INSTANCES.keys()),
            "n_spins": GPU_TIMING_N_SPINS,
            "seed": SEED,
            "cpu_device": "CPU_AER",
            "qiskit_version": qiskit.__version__,
        },
        "gpu_records": records_to_dicts(gpu_records),
        "cpu_records": records_to_dicts(cpu_records),
        "gpu_wall_time_s_total": gpu_elapsed,
        "cpu_wall_time_s_total": cpu_elapsed,
    }
    with open(gpu_results_path, "w") as f:
        json.dump(gpu_output, f, indent=2)

    print(f"Saved CPU-vs-GPU timing comparison to {os.path.relpath(gpu_results_path, repo_root)}")
    print(f"GPU total: {gpu_elapsed:.1f}s, CPU_AER total: {cpu_elapsed:.1f}s")
except Exception as exc:
    print(
        f"GPU timing comparison skipped ({type(exc).__name__}: {exc}). "
        "This is expected outside a Colab GPU runtime with qiskit-aer-gpu "
        "installed -- not an error in the main CPU sweep above."
    )

[uniform_FM] p=1 COBYLA: best_energy=-4.722070 E_0=-9.000000 epsilon=0.475326 (0.93s, 64 fevals)
[uniform_FM] p=1 L-BFGS-B: best_energy=-4.722070 E_0=-9.000000 epsilon=0.475326 (1.41s, 90 fevals)
[uniform_FM] p=2 COBYLA: best_energy=-5.420922 E_0=-9.000000 epsilon=0.397675 (2.03s, 100 fevals)
[uniform_FM] p=2 L-BFGS-B: best_energy=-4.901533 E_0=-9.000000 epsilon=0.455385 (2.04s, 145 fevals)
[uniform_FM] p=3 COBYLA: best_energy=-6.448979 E_0=-9.000000 epsilon=0.283447 (2.00s, 100 fevals)
[uniform_FM] p=3 L-BFGS-B: best_energy=-5.899819 E_0=-9.000000 epsilon=0.344465 (1.94s, 126 fevals)
[uniform_FM] p=1 COBYLA: best_energy=-4.722070 E_0=-9.000000 epsilon=0.475326 (0.74s, 68 fevals)
[uniform_FM] p=1 L-BFGS-B: best_energy=-4.722070 E_0=-9.000000 epsilon=0.475326 (0.92s, 90 fevals)
[uniform_FM] p=2 COBYLA: best_energy=-5.420922 E_0=-9.000000 epsilon=0.397675 (1.49s, 100 fevals)
[uniform_FM] p=2 L-BFGS-B: best_energy=-4.901532 E_0=-9.000000 epsilon=0.455385 (2.62s, 145 fevals)
[uniform_FM] p

## Warm-start comparison: standard QAOA vs. Egger-WS

Head-to-head comparison of three methods on the **uniform AFM
ring** (`ising_model.generate_uniform_AFM_ring`: `J=+1`, `h=0`, PBC):
standard (unwarmed) QAOA and Egger et al.
WS-QAOA (`variant="continuous"`). Calls straight into
`src.experiment.run_warm_start_comparison`/`write_warm_start_comparison_csv`,
which themselves only call `src.warm_start.egger_ws`.

Writes **one CSV per method** to `results/` (`ising_results_<method>.csv`).
Then `notebooks/analysis.ipynb` loads all of these (`results/ising_results_*.csv`).

**Before drawing conclusions from a `P_gs` comparison**: Egger-WS's
mean-field relaxation is a one-shot classical preprocessing pass costing
zero circuit evaluations before the (single) variational optimization
begins, so it is not cost-free relative to standard QAOA in wall-clock
terms even though it is free in circuit evaluations`.

ND-AWS used to be the third method here; it was removed for
being too slow to compare to the Egger WS methods.

In [ ]:
import time

from src.experiment import run_warm_start_comparison, write_warm_start_comparison_csv

WS_N_VALUES = [7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
WS_P_VALUES = [1, 2, 3, 4]
WS_DEVICE = COLAB_DEVICE          # setup cell's install choice
WS_OPTIMIZER_METHOD = "L-BFGS-B"
WS_N_RESTARTS = 5
WS_MINIMIZE_OPTIONS = None        # uncapped
WS_SEED = 2026


WS_METHODS = {
    "standard": dict(),
    "egger_continuous": dict(egger_variant="continuous"),
}

ws_csv_paths = {}
for method, extra_kwargs in WS_METHODS.items():
    print(f"\n=== Warm-start comparison: method={method} ===")
    t0 = time.time()
    rows = run_warm_start_comparison(
        n_values=WS_N_VALUES,
        p_values=WS_P_VALUES,
        method=method,
        device=WS_DEVICE,
        optimizer_method=WS_OPTIMIZER_METHOD,
        n_restarts=WS_N_RESTARTS,
        minimize_options=WS_MINIMIZE_OPTIONS,
        seed=WS_SEED,
        verbose=True,
        **extra_kwargs,
    )
    path = write_warm_start_comparison_csv(rows, filename=f"ising_results_{method}.csv")
    ws_csv_paths[method] = path
    print(f"[{method}] {len(rows)} rows in {time.time() - t0:.1f}s -> {os.path.relpath(path, repo_root)}")

print("\nWarm-start comparison CSVs written:", ws_csv_paths)